In [ ]:
# connect to postgres db

import os
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, inspect
import pandas as pd

load_dotenv(Path("../.env"))

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST', 'localhost')}:{os.getenv('DB_PORT', '5432')}/{os.getenv('DB_NAME')}"
)


In [ ]:
inspector = inspect(engine)
tables = inspector.get_table_names()
print(tables)


['properties', 'utility_bills', 'providers', 'utilities']


In [7]:
import pdfplumber
import re

def extract_utility_data(pdf_path: str) -> dict:
    """Extract key fields from a utility bill PDF."""
    result = {
        "account_number": None,
        "bill_date": None,
        "due_date": None,
        "total_due": None,
        "usage": None,
        "tables": [],
    }

    with pdfplumber.open(pdf_path) as pdf:
        full_text = ""
        for page in pdf.pages:
            text = page.extract_text() or ""
            full_text += text + "\n"

            # Extract tables from each page
            for table in page.extract_tables():
                df = pd.DataFrame(table[1:], columns=table[0])
                result["tables"].append(df)

        # Example regex patterns — adjust to match your provider's format
        if m := re.search(r"Account\s*(?:Number|#)[:\s]+(\S+)", full_text, re.I):
            result["account_number"] = m.group(1)
        if m := re.search(r"Bill\s*Date[:\s]+([\d/\-]+)", full_text, re.I):
            result["bill_date"] = m.group(1)
        if m := re.search(r"Due\s*Date[:\s]+([\d/\-]+)", full_text, re.I):
            result["due_date"] = m.group(1)
        if m := re.search(r"(?:Total\s*Due|Amount\s*Due)[:\s]+\$?([\d,\.]+)", full_text, re.I):
            result["total_due"] = float(m.group(1).replace(",", ""))
        if m := re.search(r"([\d,\.]+)\s*(?:kWh|CCF|therms|gallons)", full_text, re.I):
            result["usage"] = m.group(1)

    return result


# Usage
data = extract_utility_data("../sample_data/2026-01-15_AESIndiana.pdf")
print({k: v for k, v in data.items() if k != "tables"})
for i, table in enumerate(data["tables"]):
    print(f"\n--- Table {i+1} ---")
    print(table)


{'account_number': '200000393066', 'bill_date': None, 'due_date': '02/05/2026', 'total_due': 148.97, 'usage': None}

--- Table 1 ---
Empty DataFrame
Columns: [Amount Due, $148.97]
Index: []

--- Table 2 ---
Empty DataFrame
Columns: [Monthly Account Summary, Billing Date: 01/15/2026]
Index: []

--- Table 3 ---
                         MMeessssaaggee CCeenntteerr
0  Our records indicate that you are enrolled in ...
1  and your payment will be processed based on yo...
2  instructions. Please log into your account at ...
3  to view or update your automatic payment profi...
4  coming!Use PowerView to monitor your daily ene...
5  simple ways to save.Log in to your AES Indiana...
6                                           started.
7                                                   

--- Table 4 ---
Empty DataFrame
Columns: [Total Account Balance, $148.97]
Index: []

--- Table 5 ---
  Metered Electric and Other Services         NaN  NaN  \
0                    Service Address:         NaN  N